In [ ]:
!unzip -q bucket_data.zip -d bucket_data
!ls bucket_data

Academy  Talent


In [ ]:
import glob, json
import pandas as pd

academy = {p: pd.read_csv(p, dtype=str) for p in glob.glob("bucket_data/Academy/*.csv")}
applicants = {p: pd.read_csv(p, dtype=str) for p in glob.glob("bucket_data/Talent/*.csv")}

json_rows = []
for p in glob.glob("bucket_data/Talent/*.json"):
    with open(p) as f:
        row = json.load(f)
    row["source_file"] = p
    json_rows.append(row)
interviews = pd.DataFrame(json_rows)

sparta_texts = {p: open(p).read() for p in glob.glob("bucket_data/Talent/*.txt")}

print("Academy CSVs:  ", len(academy))
print("Applicant CSVs:", len(applicants))
print("JSONs:         ", len(interviews))
print("Sparta TXTs:   ", len(sparta_texts))

Academy CSVs:   36
Applicant CSVs: 12
JSONs:          3105
Sparta TXTs:    152


In [ ]:
import re

LINE = re.compile(
    r"^(?P<name>.+?)\s*-\s*Psychometrics:\s*(?P<psychometric_score>\d+)\s*/\s*(?P<psychometric_max>\d+)"
    r"\s*,\s*Presentation:\s*(?P<presentation_score>\d+)\s*/\s*(?P<presentation_max>\d+)\s*$"
)

rows, unreadable = [], []

for path, text in sparta_texts.items():
    lines = [l.strip() for l in text.splitlines() if l.strip()]   # drop blank lines
    date_line, location_line, people = lines[0], lines[1], lines[2:]

    for line in people:
        match = LINE.match(line)
        if match:
            rows.append({"source_file": path, "date_raw": date_line,
                         "location_raw": location_line, **match.groupdict()})
        else:
            unreadable.append((path, line))

raw_sparta_df = pd.DataFrame(rows)    # raw copy (dont edit)
print(len(raw_sparta_df), "people read")
print(len(unreadable), "lines not understood")
for u in unreadable[:10]:
    print("  ", u)
raw_sparta_df.head()

4134 people read
0 lines not understood


,source_file,date_raw,location_raw,name,psychometric_score,psychometric_max,presentation_score,presentation_max
0,bucket_data/Talent/Sparta Day 19 March 2019.txt,Tuesday 19 March 2019,London Academy,TONIA GOODIE,57,100,21,32
1,bucket_data/Talent/Sparta Day 19 March 2019.txt,Tuesday 19 March 2019,London Academy,CORT TROW,62,100,10,32
2,bucket_data/Talent/Sparta Day 19 March 2019.txt,Tuesday 19 March 2019,London Academy,HENRIK MOTT,56,100,19,32
3,bucket_data/Talent/Sparta Day 19 March 2019.txt,Tuesday 19 March 2019,London Academy,WILLY MURPHEY,52,100,22,32
4,bucket_data/Talent/Sparta Day 19 March 2019.txt,Tuesday 19 March 2019,London Academy,JEROMY MEBIUS,61,100,25,32


In [11]:
all_applicants = pd.concat(applicants.values(), ignore_index=True)

print(all_applicants["invited_by"].value_counts(dropna=False))
print("\nDifferent spellings:", all_applicants["invited_by"].nunique())

invited_by
Rupert Ripple       716
Doris Bellasis      593
Gismo Tilling       575
Stacey Broad        560
Bruno Bellbrook     557
NaN                 557
Sunny Sladefield    532
Fifi Etton          286
Fifi Eton           271
Bruno Belbrook       44
Name: count, dtype: int64

Different spellings: 9


In [ ]:
df = raw_sparta_df.copy()

# 1. Score maximums: should only be 100 and 32
print("Psychometric max values:", df["psychometric_max"].unique())
print("Presentation max values: ", df["presentation_max"].unique())

# 2. Dates: do they all convert?
df["date"] = pd.to_datetime(df["date_raw"], format="%A %d %B %Y", errors="coerce")
print("\nDates that didn't convert:", df.loc[df["date"].isna(), "date_raw"].unique())
print("Date range:", df["date"].min(), "to", df["date"].max())

# 3. Does the date in the file name match line 1?
file_date = pd.to_datetime(df["source_file"].str.extract(r"(\d{1,2} \w+ \d{4})")[0],
                           format="%d %B %Y", errors="coerce")
print("Rows where filename date doesn't match:", (file_date != df["date"]).sum())

# 4. Locations
print("\nLocations:")
print(df["location_raw"].value_counts())

# 5. Scores above the maximum
print("\nScores above max:",
      ((df["psychometric_score"].astype(int) > df["psychometric_max"].astype(int)) |
       (df["presentation_score"].astype(int) > df["presentation_max"].astype(int))).sum())

# 6. Duplicates
print("Same name on the same day:", df.duplicated(subset=["name", "date_raw"]).sum())
print("Same name on different days:", df["name"].duplicated(keep=False).sum() -
      df.duplicated(subset=["name", "date_raw"], keep=False).sum())

Psychometric max values: ['100']
Presentation max values:  ['32']

Dates that didn't convert: []
Date range: 2019-01-02 00:00:00 to 2019-12-19 00:00:00
Rows where filename date doesn't match: 0

Locations:
location_raw
London Academy        2782
Birmingham Academy    1352
Name: count, dtype: int64

Scores above max: 0
Same name on the same day: 0
Same name on different days: 4


In [ ]:
same_name = df[df["name"].duplicated(keep=False)].sort_values("name")
same_name[["name", "date", "location_raw", "psychometric_score", "presentation_score"]]

,name,date,location_raw,psychometric_score,presentation_score
588,NICOLETTE BONEHILL,2019-11-19,London Academy,47,20
3458,NICOLETTE BONEHILL,2019-02-05,London Academy,52,23
533,SHURLOCKE CRINGLE,2019-05-21,London Academy,52,21
1674,SHURLOCKE CRINGLE,2019-01-31,Birmingham Academy,49,18


In [ ]:
all_applicants = pd.concat(
    [d.assign(source_file=p) for p, d in applicants.items()], ignore_index=True)

for person in ["Nicolette Bonehill", "Shurlocke Cringle"]:
    print("=" * 20, person)
    print("Applicants:")
    print(all_applicants[all_applicants["name"].str.strip().str.lower() == person.lower()]
          [["name", "dob", "email", "invited_date", "month", "source_file"]].to_string())
    print("Interviews:")
    print(interviews[interviews["name"].str.strip().str.lower() == person.lower()]
          [["name", "date", "result", "source_file"]].to_string())

==================== Nicolette Bonehill
Applicants:
                    name         dob                     email invited_date          month                               source_file
2400  Nicolette Bonehill  14/08/1999       neebee9@addthis.com           19  NOVEMBER 2019  bucket_data/Talent/Nov2019Applicants.csv
3633  Nicolette Bonehill  05/04/1995  nbonehill1g@hubpages.com           05  February 2019  bucket_data/Talent/Feb2019Applicants.csv
Interviews:
                    name        date result                    source_file
1603  Nicolette Bonehill  05/02/2019   Fail  bucket_data/Talent/11522.json
==================== Shurlocke Cringle
Applicants:
                   name         dob                 email invited_date         month                               source_file
15    Shurlocke Cringle  27/11/1995                   NaN           21      May 2019  bucket_data/Talent/May2019Applicants.csv
2045  Shurlocke Cringle  16/04/1992  scringle8@uol.com.br           31  January 20

In [ ]:
# names: collapse spaces, trim, normal case
df["name"] = (df["name"].str.replace(r"\s+", " ", regex=True)
                        .str.strip()
                        .str.title())                 # DORIE SPEARES -> Dorie Speares
df["name_key"] = df["name"].str.lower()               # used for matching later

# location: drop the word "Academy"
df["location"] = df["location_raw"].str.replace("Academy", "", regex=False).str.strip()

# scores: text -> numbers
for col in ["psychometric_score", "presentation_score"]:
    df[col] = df[col].astype(int)

sparta_clean = df[["name", "name_key", "date", "location",
                   "psychometric_score", "presentation_score", "source_file"]]

print(sparta_clean.shape)
print(sparta_clean.dtypes)
sparta_clean.head()

(4134, 7)
name                          object
name_key                      object
date                  datetime64[ns]
location                      object
psychometric_score             int64
presentation_score             int64
source_file                   object
dtype: object


,name,name_key,date,location,psychometric_score,presentation_score,source_file
0,Tonia Goodie,tonia goodie,2019-03-19,London,57,21,bucket_data/Talent/Sparta Day 19 March 2019.txt
1,Cort Trow,cort trow,2019-03-19,London,62,10,bucket_data/Talent/Sparta Day 19 March 2019.txt
2,Henrik Mott,henrik mott,2019-03-19,London,56,19,bucket_data/Talent/Sparta Day 19 March 2019.txt
3,Willy Murphey,willy murphey,2019-03-19,London,52,22,bucket_data/Talent/Sparta Day 19 March 2019.txt
4,Jeromy Mebius,jeromy mebius,2019-03-19,London,61,25,bucket_data/Talent/Sparta Day 19 March 2019.txt


In [ ]:
sparta_clean.isna().sum()

,0
name,0
name_key,0
date,0
location,0
psychometric_score,0
presentation_score,0
source_file,0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

sparta_clean.to_csv("/content/drive/MyDrive/sparta_day_clean.csv", index=False)
print("saved")

Mounted at /content/drive
saved
